# 🏥 MediQwen — Notebook 01: Knowledge Base Setup & RAG Pipeline Validation

This notebook covers:
- **Environment & dependency check**
- **Knowledge base ingestion** via `MedicalKnowledgeBase`
- **Vector store population** via `MedicalVectorStore`
- **RAG pipeline validation** — similarity search, hybrid search, retriever


## 🔧 Path Setup & Project Root

In [ ]:
import sys
import os
from pathlib import Path

# ── Adjust this to your actual project root ──────────────────────────────────
PROJECT_ROOT = Path("../").resolve()   # notebooks/ is one level below root
#sys.path.insert(0, str(PROJECT_ROOT))

print(f"📁 Project root  : {PROJECT_ROOT}")
print(f"📁 Working dir   : {Path.cwd()}")
print(f"🐍 Python version: {sys.version.split()[0]}")

# Validate key source files exist
required_files = [
    PROJECT_ROOT / "src" / "rag" / "knowledge_base.py",
    PROJECT_ROOT / "src" / "rag" / "vector_store.py",
    PROJECT_ROOT / "config" / "settings.py",
    PROJECT_ROOT / "config" / "skills.md",
    PROJECT_ROOT / "src" / "agent" / "graph.py",
    PROJECT_ROOT / "src" / "agent" / "nodes.py",
    PROJECT_ROOT / "src" / "agent" / "state.py",
    PROJECT_ROOT / "src" / "tools" / "utils.py",
    PROJECT_ROOT / "src" / "tools" / "web_scraper.py",
    PROJECT_ROOT / "src" / "tools" / "system_prompt.py",
    PROJECT_ROOT / "src" / "preprocessing" / "image_processor.py",
    PROJECT_ROOT / "src" / "models" / "embeddings.py",
    PROJECT_ROOT / "src" / "safety" / "guardrails.py",
    PROJECT_ROOT / "src" / "safety" / "risk_classifier.py",
    PROJECT_ROOT / "src" / "rag" / "knowledge_base.py",
    PROJECT_ROOT / "src" / "rag" / "vector_store.py",
    PROJECT_ROOT / "app" / "api_server.py",
    PROJECT_ROOT / "app" / "streamlit_interface.py",
]
for f in required_files:
    status = "✅" if f.exists() else "❌ MISSING"
    print(f"  {status}  {f.relative_to(PROJECT_ROOT)}")

📁 Project root  : C:\santosh\MediGemma
📁 Working dir   : c:\santosh\MediGemma\notebooks
🐍 Python version: 3.14.2
  ✅  src\rag\knowledge_base.py
  ✅  src\rag\vector_store.py
  ✅  config\settings.py


## 🔍 — Library Version Check

In [ ]:
import importlib

packages = {
    "chromadb": "chromadb",
    "langchain": "langchain",
    "langchain_community": "langchain_community",
    "langchain_huggingface": "langchain_huggingface",
    "sentence_transformers": "sentence_transformers",
    "torch": "torch",
    
}

print("📦 Installed package versions:")
print("-" * 40)
all_ok = True
for display_name, import_name in packages.items():
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, "__version__", "unknown")
        print(f"  ✅ {display_name:<30} {version}")
    except ImportError:
        print(f"  ❌ {display_name:<30} NOT INSTALLED")
        all_ok = False

print("-" * 40)
print("✅ All packages ready" if all_ok else "❌ Some packages missing — re-run Cell 1")

📦 Installed package versions:
----------------------------------------
  ✅ chromadb                       1.5.9
  ✅ langchain                      1.3.1
  ✅ langchain_community            0.4.2


c:\santosh\MediGemma\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ✅ langchain_huggingface          unknown
  ✅ sentence_transformers          5.5.1
  ✅ torch                          2.12.0+cpu
----------------------------------------
✅ All packages ready


## 📚 Ingest Documents into Knowledge Base

In [ ]:
import os
from pathlib import Path
from rag.knowledge_base import MedicalKnowledgeBaseBuilder
from rag.vector_store import MedicalVectorStore

# Define your project directory tracking path
KB_ROOT = Path(PROJECT_ROOT) / "data" / "knowledge_base"

print("🏥 Starting MediQwen Nested Directory Ingestion Loop...")

if not KB_ROOT.exists():
    print(f"❌ Error: Data directory target '{KB_ROOT}' does not exist on disk.")
else:
    # 1. Discover all markdown files across all subfolders recursively
    target_files = list(KB_ROOT.rglob("*.md"))
    
    if target_files:
        print(f"🔍 Discovered {len(target_files)} condition markdown file(s) scheduled for parsing.")
        
        # 2. Initialize clean builder
        kb_builder = MedicalKnowledgeBaseBuilder()
        
        # 3. Loop through every discovered file one by one
        for file_path in target_files:
            print(f"  📄 Processing: {file_path.relative_to(KB_ROOT)}")
            kb_builder.parse_markdown(file_path)
            
        # Print summary statistics across all ingested markdown data chunks
        kb_builder.print_statistics()
        
        # 4. Initialize Vector Store and stream everything directly into ChromaDB in one batch
        print("\n📥 Streaming all parsed chunks into local ChromaDB collection...")
        vector_store = MedicalVectorStore()
        
        # Retrieve all formatted chunks from the knowledge base builder and add them to the vector store
        all_chunks = kb_builder.get_all_formatted_chunks()
        success = vector_store.add_documents(all_chunks)
        
        if success:
            print("✅ Database synchronization complete!")
            # Print updated stats straight from ChromaDB's persistence layer to verify
            db_stats = vector_store.get_collection_stats()
            print(f"📊 Current database total chunk count: {db_stats.get('count', 0)}")
        else:
            print("❌ Failed to commit some chunks to the Vector Store.")
            
    else:  
        print(f"⚠️ No markdown files (.md) found inside '{KB_ROOT}' or its subfolders.")

c:\santosh\MediGemma\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-21 21:18:40,984 - INFO - Initializing Persistent ChromaDB client at: ./chroma_db


🏥 Starting MediGemma Nested Directory Ingestion Loop...
🔍 Discovered 82 condition markdown file(s) scheduled for parsing.
  📄 Processing: cardiovascular\angina.md
  📄 Processing: cardiovascular\cardiomyopathy.md
  📄 Processing: cardiovascular\chest_pain.md
  📄 Processing: cardiovascular\coronary_heart_diesease.md
  📄 Processing: cardiovascular\heart_attack.md
  📄 Processing: cardiovascular\high_blood_pressure.md
  📄 Processing: cardiovascular\low_blood_pressure.md
  📄 Processing: cardiovascular\mini_stroke.md
  📄 Processing: cardiovascular\stoke.md
  📄 Processing: digestive\acid_reflux.md
  📄 Processing: digestive\constipation.md
  📄 Processing: digestive\crohn's_disease.md
  📄 Processing: digestive\gallstones.md
  📄 Processing: digestive\gastritis.md
  📄 Processing: digestive\indigestion.md
  📄 Processing: digestive\irritable_bowel_syndrome.md
  📄 Processing: digestive\ulcerative_colitis.md
  📄 Processing: first_aid\first_aid.md
  📄 Processing: health\family_planning.md
  📄 Processing

2026-06-21 21:19:14,748 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-06-21 21:19:14,771 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-06-21 21:19:15,030 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-06-21 21:19:15,050 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-06-21 21:19:15,077 - INFO - Loading SentenceTransformer model from BAAI/bge-large-en-v1.5.
2026-06-21 21:19:15,296 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 3

✅ Database synchronization complete!
📊 Current database total chunk count: 1038


## 🔍 RAG Pipeline Validation: Similarity Search

In [ ]:
# Test queries covering each knowledge category
TEST_QUERIES = [

    # Symptoms
    ("high fever and chills", None),
    ("severe headache", None),
    ("persistent cough", None),
    ("abdominal pain after eating", None),

    # Respiratory
    ("asthma attack treatment", None),
    ("pneumonia symptoms", None),

    # Cardiovascular
    ("hypertension treatment", None),
    ("heart attack emergency", None),

    # First Aid
    ("first aid burn treatment steps", None),
    ("nosebleed first aid", None),

    # Pediatric
    ("child fever dehydration", None),

    # Infectious Disease
    ("malaria symptoms", None),
    ("tuberculosis treatment", None),
]

print("=" * 65)
print("🔍 SIMILARITY SEARCH VALIDATION")
print("=" * 65)

all_passed = True
for query, where_filter in TEST_QUERIES:
    filter_label = f" [filter: {where_filter}]" if where_filter else ""
    print(f"\n📝 Query: '{query}'{filter_label}")
    print("-" * 55)
    
    results = vs.similarity_search(query, k=3, where=where_filter)
    
    if not results:
        print("  ❌ No results returned")
        all_passed = False
        continue
    
    for i, doc in enumerate(results):
        print(f"  [{i+1}] Source : {doc.metadata.get('source', 'unknown')}")
        print(f"       Domain : {doc.metadata.get('domain', 'unknown')}")
        print(f"       Trust  : {doc.metadata.get('trust_level', '?')}")
        print(f"       Title  : {doc.metadata.get('doc_title')}")
        print(f"       Text   : {doc.page_content[:100].strip()}...")
        print()

print("=" * 65)
print("✅ Similarity search validation PASSED" if all_passed else "⚠️  Some queries returned no results")

🔍 SIMILARITY SEARCH VALIDATION

📝 Query: 'hypertension blood pressure treatment'
-------------------------------------------------------
  [1] Source : WHO
       Domain : cardiovascular
       Trust  : 10
       Text   : World Health Organization — Hypertension Fact Sheet

Hypertension (hypertension) is a major risk fac...

  [2] Source : WHO
       Domain : general_medicine
       Trust  : 10
       Text   : BLEEDING
1. Apply direct pressure with a clean cloth
2. Elevate the injured limb above heart level i...

  [3] Source : Mayo Clinic
       Domain : pharmacology
       Trust  : 8
       Text   : Mayo Clinic — Common Medications Reference Guide

PARACETAMOL (ACETAMINOPHEN)
- Uses: Pain relief, f...


📝 Query: 'chest pain heart attack emergency symptoms'
-------------------------------------------------------
  [1] Source : NHS
       Domain : cardiovascular
       Trust  : 8
       Text   : National Health Service — Chest Pain Assessment Guide

Chest pain requires immediate evalua

## 🔀 RAG Pipeline Validation: Hybrid Search

In [ ]:
print("=" * 65)
print("🔀 HYBRID SEARCH (MMR) VALIDATION")
print("=" * 65)


hybrid_results = vs.hybrid_search(
    query,
    k=4,
    lambda_mult=0.7   # 0.7 = favour relevance; 0.3 = favour diversity
)

if hybrid_results:
    print(f"✅ Returned {len(hybrid_results)} diverse results (MMR)")
    print()
    for i, doc in enumerate(hybrid_results):
        print(f"  [{i+1}] Source : {doc.metadata.get('source', 'unknown')}")
        print(f"       Domain : {doc.metadata.get('domain', 'unknown')}")
        print(f"       Text   : {doc.page_content[:110].strip()}...")
        print()
    
    # Diversity check: confirm sources are not all identical
    sources = [doc.metadata.get('source') for doc in hybrid_results]
    unique_sources = len(set(sources))
    print(f"📊 Source diversity: {unique_sources}/{len(hybrid_results)} unique sources")
    diversity_ok = unique_sources > 1
    print("✅ Good diversity" if diversity_ok else "⚠️ Low diversity — consider lowering lambda_mult")
else:
    print("❌ Hybrid search returned no results")

🔀 HYBRID SEARCH (MMR) VALIDATION

📝 Query: 'emergency symptoms requiring immediate medical attention'
-------------------------------------------------------
✅ Returned 4 diverse results (MMR)

  [1] Source : CDC
       Domain : infectious_disease
       Text   : CDC — Sepsis Early Recognition & Triage Protocol

Sepsis is a life-threatening emergency caused by the body's...

  [2] Source : NHS
       Domain : cardiovascular
       Text   : National Health Service — Chest Pain Assessment Guide

Chest pain requires immediate evaluation. Do not ignore...

  [3] Source : WHO
       Domain : general_medicine
       Text   : BLEEDING
1. Apply direct pressure with a clean cloth
2. Elevate the injured limb above heart level if possible...

  [4] Source : CDC
       Domain : diabetes
       Text   : Centers for Disease Control — Diabetes Symptoms Guide

Type 2 Diabetes Mellitus: Symptoms & Early Detection

E...

📊 Source diversity: 3/4 unique sources
✅ Good diversity


## 🤖 RAG Pipeline Validation: LangChain Retriever

In [ ]:
print("=" * 65)
print("🤖 LANGCHAIN RETRIEVER VALIDATION")
print("=" * 65)

# Get retriever with MMR settings
retriever = vs.get_retriever(search_kwargs={"k": 4, "fetch_k": 12, "lambda_mult": 0.6})

retriever_query = "steps to treat severe bleeding wound"
print(f"\n📝 Retriever query: '{retriever_query}'")
print("-" * 55)

try:
    # LangChain retriever uses .invoke()
    retrieved_docs = retriever.invoke(retriever_query)
    
    print(f"✅ Retrieved {len(retrieved_docs)} documents via retriever")
    print()
    
    for i, doc in enumerate(retrieved_docs):
        print(f"  [{i+1}] Source : {doc.metadata.get('source', 'unknown')}")
        print(f"       Text   : {doc.page_content[:120].strip()}...")
        print()
        
except Exception as e:
    print(f"❌ Retriever failed: {e}")

2026-06-01 21:52:32,174 - INFO - ✅ Retriever created with MMR search


🤖 LANGCHAIN RETRIEVER VALIDATION

📝 Retriever query: 'steps to treat severe bleeding wound'
-------------------------------------------------------
✅ Retrieved 4 documents via retriever

  [1] Source : WHO
       Text   : BLEEDING
1. Apply direct pressure with a clean cloth
2. Elevate the injured limb above heart level if possible
3. Mainta...

  [2] Source : WHO
       Text   : World Health Organization — Basic First Aid Protocols

BURNS
1. Cool the burn with cool (not upper respiratory tract inf...

  [3] Source : WHO
       Text   : World Health Organization — Emergency Triage Reference (START Protocol)

TRIAGE CATEGORIES

RED (Immediate) — Life-threa...

  [4] Source : MedlinePlus
       Text   : MedlinePlus — Antibiotic Use Guidelines

Antibiotics treat bacterial infections — they DO NOT work against viruses.

Whe...

